# Reconhecimento dos integrantes do grupo com CNN

Treinamos **do zero** uma Rede Neural Convolucional que recebe a foto de um rosto e diz qual integrante do grupo ela é.

**O que é feito por nós e o que vem de biblioteca:**

| Etapa | Ferramenta |
|---|---|
| Abrir o arquivo .jpg/.png | `PIL` (só transforma o arquivo em uma matriz de números) |
| Recorte, tons de cinza, redimensionamento, equalização, aumento de dados, divisão dos dados | **código próprio com NumPy** |
| Montar e treinar a CNN | TensorFlow/Keras, **sem modelo pré-treinado** |
| Matriz de confusão e métricas | **código próprio com NumPy** |
| Gráficos | `matplotlib` |

**Etapas do notebook**

1. Configuração
2. Coleta das fotos
3. Pré-processamento
4. Divisão em treino, validação e teste
5. Aumento de dados
6. Montagem da CNN
7. Treinamento
8. Avaliação: matriz de confusão e métricas
9. Imagens intermediárias da rede
10. Teste com foto nova e reconhecimento ao vivo pela webcam
11. Perguntas prováveis na apresentação

> Antes de rodar: **Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)**.

## 1. Configuração

In [ ]:
# ---------------------------------------------------------------
# Bibliotecas
# ---------------------------------------------------------------
import os, io, base64, zipfile             # arquivos e conversões (já vêm com o Python)
import numpy as np                         # matrizes: usado em todo o tratamento das imagens e nas métricas
import matplotlib.pyplot as plt            # gráficos e exibição das imagens
from PIL import Image                      # apenas para abrir o arquivo de imagem
import tensorflow as tf                    # base para a rede neural
from tensorflow import keras               # montagem e treino da CNN
from tensorflow.keras import layers        # camadas da CNN (Conv2D, MaxPooling2D, Dense...)

# Verifica se estamos rodando no Google Colab
try:
    import google.colab
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

print("TensorFlow", tf.__version__, "| GPU disponível:", len(tf.config.list_physical_devices("GPU")) > 0)

In [ ]:
# ---------------------------------------------------------------
# Configurações gerais (é só aqui que precisamos mexer)
# ---------------------------------------------------------------
DATASET_DIR       = "/content/drive/MyDrive/cnn_reconhecimento/dataset"  # uma subpasta por integrante
IMG_SIZE          = 96    # toda imagem vira 96 x 96 pixels
FOTOS_POR_PESSOA  = 80    # fotos tiradas a cada captura pela webcam
COPIAS_AUMENTADAS = 3     # quantas versões modificadas de cada foto de treino vamos criar
EPOCAS            = 60    # número máximo de passadas pelos dados de treino
BATCH             = 32    # quantas fotos a rede vê antes de cada ajuste dos pesos
SEED              = 42    # semente aleatória: garante o mesmo resultado a cada execução

np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Coleta das fotos

Cada integrante tem a sua pasta, e **o nome da pasta é o rótulo** que a rede vai aprender:

```
dataset/
├── ana/     ana_0000.jpg, ana_0001.jpg, ...
├── bruno/   ...
└── carla/   ...
```

Salvamos tudo no **Google Drive**, porque os arquivos do Colab são apagados quando a sessão termina.

**Dicas para a rede aprender o rosto, e não o fundo:**
- Façam 3 ou mais capturas por pessoa, em lugares e iluminações diferentes.
- Mantenham o rosto centralizado e mexam levemente a cabeça durante a captura.
- Usem quantidades parecidas de fotos por pessoa.

In [ ]:
# Conecta o Google Drive e cria a pasta do dataset
if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

os.makedirs(DATASET_DIR, exist_ok=True)
print("As fotos ficam em:", DATASET_DIR)

### Opção A: captura pela webcam

O Colab roda no servidor do Google, que não tem acesso à sua câmera. Por isso usamos um pequeno código **JavaScript**, que roda no seu navegador: ele liga a câmera, tira as fotos e as envia ao Python como texto (base64).

In [ ]:
# Código JavaScript executado no navegador para usar a webcam
JS_WEBCAM = '''
async function capturar(n, rotulo) {
  // 1. Cria na página um texto de aviso e o vídeo da câmera
  const area  = document.createElement('div');
  const texto = document.createElement('p');
  const video = document.createElement('video');
  area.append(texto, video);
  document.body.appendChild(area);

  // 2. Liga a webcam e mostra a imagem ao vivo
  const stream = await navigator.mediaDevices.getUserMedia({video: true});
  video.srcObject = stream;
  await video.play();
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

  // 3. Espera 3 segundos para a pessoa se posicionar
  texto.textContent = rotulo + ': centralize o rosto. Começando em 3 segundos...';
  await new Promise(r => setTimeout(r, 3000));

  // 4. Tira n fotos, uma a cada 250 ms, e guarda cada uma como JPEG em texto (base64)
  const canvas = document.createElement('canvas');
  canvas.width = video.videoWidth;
  canvas.height = video.videoHeight;
  const fotos = [];
  for (let i = 0; i < n; i++) {
    canvas.getContext('2d').drawImage(video, 0, 0);
    fotos.push(canvas.toDataURL('image/jpeg', 0.9));
    texto.textContent = rotulo + ': foto ' + (i + 1) + ' de ' + n + ' (mexa levemente a cabeça)';
    await new Promise(r => setTimeout(r, 250));
  }

  // 5. Desliga a câmera, remove o vídeo da página e devolve as fotos para o Python
  stream.getVideoTracks()[0].stop();
  area.remove();
  return fotos;
}
'''

def tirar_fotos(n, rotulo):
    # Executa o JavaScript e devolve uma lista de imagens (matrizes de pixels)
    from google.colab.output import eval_js
    from IPython.display import display, Javascript
    display(Javascript(JS_WEBCAM))                                   # envia o código para o navegador
    textos = eval_js(f'capturar({n}, "{rotulo}")', timeout_sec=600)  # roda e espera as fotos voltarem
    imagens = []
    for t in textos:
        bytes_jpg = base64.b64decode(t.split(",")[1])                # tira o cabeçalho "data:image/jpeg;base64,"
        imagens.append(np.array(Image.open(io.BytesIO(bytes_jpg))))  # bytes do JPEG -> matriz de pixels
    return imagens

def capturar_fotos(nome, n=FOTOS_POR_PESSOA):
    # Tira n fotos de um integrante e salva na pasta dele
    pasta = os.path.join(DATASET_DIR, nome)
    os.makedirs(pasta, exist_ok=True)
    ja_existem = len(os.listdir(pasta))                              # continua a numeração se já houver fotos
    for i, img in enumerate(tirar_fotos(n, nome)):
        Image.fromarray(img).save(os.path.join(pasta, f"{nome}_{ja_existem + i:04d}.jpg"))
    print(f"{n} fotos salvas em {pasta}")

In [ ]:
# Uma pessoa por vez na frente da câmera: descomentem a linha e rodem a célula.
# Rodem de novo em outro lugar/iluminação para criar mais variedade.

# capturar_fotos("integrante_1")
# capturar_fotos("integrante_2")
# capturar_fotos("integrante_3")
# capturar_fotos("integrante_4")

### Opção B: enviar um `.zip`

O `.zip` deve conter **diretamente** as pastas dos integrantes (`ana/`, `bruno/`...). Também é possível copiar as pastas direto para o Drive, sem usar o zip.

In [ ]:
ENVIAR_ZIP = False   # troque para True e rode a célula

if EM_COLAB and ENVIAR_ZIP:
    from google.colab import files
    for nome_zip, conteudo in files.upload().items():        # abre a janela para escolher o arquivo
        zipfile.ZipFile(io.BytesIO(conteudo)).extractall(DATASET_DIR)
        print("Extraído:", nome_zip)

### Leitura das fotos

Percorremos as pastas e montamos duas listas: o caminho de cada foto e o número da pessoa (0, 1, 2...).

Celulares às vezes salvam a foto "deitada" e anotam a rotação certa nos metadados (EXIF). Nesse caso, desviramos a matriz.

In [ ]:
def ler_imagem(caminho):
    # Abre o arquivo e devolve uma matriz (altura, largura, 3) com os valores R, G, B de cada pixel
    img = Image.open(caminho)
    orientacao = img.getexif().get(274, 1)    # 274 = código do EXIF que guarda a orientação
    pixels = np.array(img.convert("RGB"))
    if orientacao == 3: pixels = np.rot90(pixels, 2)    # foto de cabeça para baixo -> gira 180°
    if orientacao == 6: pixels = np.rot90(pixels, -1)   # foto deitada -> gira 90° no sentido horário
    if orientacao == 8: pixels = np.rot90(pixels, 1)    # foto deitada -> gira 90° no sentido anti-horário
    return pixels

# Cada subpasta é um integrante (ignoramos pastas ocultas e de sistema, que começam com "." ou "_")
CLASSES = sorted(p for p in os.listdir(DATASET_DIR)
                 if os.path.isdir(os.path.join(DATASET_DIR, p)) and p[0] not in "._")
N_CLASSES = len(CLASSES)

caminhos, rotulos = [], []
for numero, nome in enumerate(CLASSES):
    pasta = os.path.join(DATASET_DIR, nome)
    for arquivo in sorted(os.listdir(pasta)):            # sorted = ordem em que as fotos foram tiradas
        if arquivo.lower().endswith((".jpg", ".jpeg", ".png")):
            caminhos.append(os.path.join(pasta, arquivo))
            rotulos.append(numero)
rotulos = np.array(rotulos)

assert N_CLASSES >= 2, "São necessários pelo menos 2 integrantes com fotos."
for numero, nome in enumerate(CLASSES):
    print(f"{numero} = {nome}: {np.sum(rotulos == numero)} fotos")

In [ ]:
# Mostra 6 fotos originais de cada integrante, sem nenhum tratamento
fig, eixos = plt.subplots(N_CLASSES, 6, figsize=(12, 2.2 * N_CLASSES), squeeze=False)
for linha, nome in enumerate(CLASSES):
    fotos_da_pessoa = [c for c, r in zip(caminhos, rotulos) if r == linha]
    for coluna in range(6):
        ax = eixos[linha, coluna]
        ax.axis("off")
        if coluna < len(fotos_da_pessoa):
            ax.imshow(ler_imagem(fotos_da_pessoa[coluna * len(fotos_da_pessoa) // 6]))
    eixos[linha, 0].set_title(nome, loc="left")
plt.suptitle("Fotos originais")
plt.tight_layout()
plt.show()

## 3. Pré-processamento

Toda foto passa pelas mesmas 5 etapas, cada uma com uma função simples:

| Etapa | O que faz | Por quê |
|---|---|---|
| 1. Recorte central | corta o maior quadrado do centro da foto | a CNN exige entrada quadrada de tamanho fixo; cortar não deforma o rosto |
| 2. Tons de cinza | `0,299·R + 0,587·G + 0,114·B` | 1 canal em vez de 3; a identidade está na forma, não na cor |
| 3. Redimensionamento | média de blocos + vizinho mais próximo até 96×96 | menos pixels = treino rápido; a média evita serrilhado |
| 4. Equalização de histograma | espalha os tons de cinza por toda a faixa 0–255 | fotos claras e escuras ficam parecidas |
| 5. Escala 0–1 | divide por 255 | redes neurais treinam melhor com números pequenos |

In [ ]:
def recorte_central(img):
    # Etapa 1: recorta o maior quadrado possível do centro da imagem
    altura, largura = img.shape[:2]
    lado = min(altura, largura)                  # o lado do quadrado é a menor dimensão
    topo = (altura - lado) // 2                  # quanto sobra em cima
    esquerda = (largura - lado) // 2             # quanto sobra à esquerda
    return img[topo:topo + lado, esquerda:esquerda + lado]


def para_cinza(img):
    # Etapa 2: média ponderada dos canais R, G e B (pesos da sensibilidade do olho humano)
    return 0.299 * img[:, :, 0] + 0.587 * img[:, :, 1] + 0.114 * img[:, :, 2]


def redimensionar(img, tamanho):
    # Etapa 3: reduz uma imagem quadrada 2D para tamanho x tamanho
    fator = img.shape[0] // tamanho              # quantos pixels originais cabem em 1 pixel novo
    if fator > 1:
        # (a) média de blocos: cada bloco fator x fator vira 1 pixel (a média dele)
        n = img.shape[0] // fator
        img = img[:n * fator, :n * fator]        # descarta as sobras para a divisão ficar exata
        img = img.reshape(n, fator, n, fator).mean(axis=(1, 3))
    # (b) vizinho mais próximo: para cada pixel novo, pega o pixel original correspondente
    indices = (np.arange(tamanho) * img.shape[0] / tamanho).astype(int)
    return img[indices][:, indices]


def equalizar_histograma(img):
    # Etapa 4: redistribui os tons de cinza usando o histograma acumulado
    img = img.astype(np.uint8)                              # tons inteiros de 0 a 255
    histograma = np.bincount(img.ravel(), minlength=256)    # quantos pixels existem de cada tom
    acumulado = histograma.cumsum()                         # quantos pixels têm tom <= t
    acumulado = (acumulado - acumulado.min()) / max(acumulado.max() - acumulado.min(), 1) * 255
    return acumulado[img]                                   # troca cada tom pelo seu valor acumulado


def preprocessar(img, mostrar_etapas=False):
    # Aplica as 5 etapas em sequência
    e1 = recorte_central(img)
    e2 = para_cinza(e1)
    e3 = redimensionar(e2, IMG_SIZE)
    e4 = equalizar_histograma(e3)
    e5 = e4 / 255.0                                          # Etapa 5: valores entre 0 e 1
    final = e5.reshape(IMG_SIZE, IMG_SIZE, 1).astype(np.float32)   # a CNN espera (altura, largura, canais)
    if mostrar_etapas:
        return final, [img, e1, e2, e3, e4]
    return final

In [ ]:
# Imagens intermediárias do pré-processamento: uma foto de cada integrante
titulos = ["original", "1. recorte", "2. cinza", "3. 96x96", "4. equalizada"]

for numero, nome in enumerate(CLASSES):
    primeira_foto = caminhos[list(rotulos).index(numero)]
    _, etapas = preprocessar(ler_imagem(primeira_foto), mostrar_etapas=True)

    fig, eixos = plt.subplots(1, 6, figsize=(17, 3))
    for ax, img, titulo in zip(eixos, etapas, titulos):
        ax.imshow(img.astype(np.uint8), cmap="gray")
        ax.set_title(f"{titulo}\n{img.shape[1]}x{img.shape[0]}")
        ax.axis("off")
    # Histograma: quantos pixels há de cada tom antes e depois da equalização
    eixos[5].hist(etapas[3].ravel(), bins=50, alpha=0.6, label="antes")
    eixos[5].hist(etapas[4].ravel(), bins=50, alpha=0.6, label="equalizada")
    eixos[5].set_title("histograma dos tons")
    eixos[5].legend()
    plt.suptitle(nome)
    plt.tight_layout()
    plt.show()

In [ ]:
# Pré-processa todas as fotos e junta em uma única matriz X
X = np.array([preprocessar(ler_imagem(c)) for c in caminhos])
y = rotulos
print("X:", X.shape, "-> (quantidade de fotos, altura, largura, canais)")
print("y:", y.shape, "-> número do integrante de cada foto")

## 4. Divisão em treino, validação e teste

| Conjunto | Uso | Proporção |
|---|---|---|
| Treino | a rede ajusta os pesos olhando essas fotos | ~70% |
| Validação | acompanha o treino e decide quando parar | ~15% |
| Teste | avaliação final, usado **uma única vez** | ~15% |

**Cuidado:** fotos tiradas em sequência são quase idênticas. Se uma fosse para o treino e a vizinha para o teste, a rede "já teria visto" a foto do teste e a nota ficaria falsamente alta.

Por isso agrupamos as fotos de cada pessoa em **blocos de até 10 consecutivas** e sorteamos blocos inteiros para cada conjunto. Fazemos isso por pessoa, para que todos apareçam nos três conjuntos. Se alguém tiver poucas fotos, os blocos ficam menores, para garantir que sobrem blocos para o treino.

In [ ]:
gerador = np.random.default_rng(SEED)
treino, validacao, teste = [], [], []

for classe in range(N_CLASSES):
    indices = np.where(y == classe)[0]                                # fotos desta pessoa, em ordem de captura
    assert len(indices) >= 3, f"{CLASSES[classe]} tem só {len(indices)} foto(s): são necessárias pelo menos 3"

    # Blocos de até 10 fotos seguidas. Com poucas fotos, os blocos diminuem para formar pelo menos ~7 blocos
    # (senão poderia não sobrar nenhum bloco para o treino)
    tamanho_bloco = min(10, max(1, len(indices) // 7))
    blocos = [indices[i:i + tamanho_bloco] for i in range(0, len(indices), tamanho_bloco)]
    blocos = [blocos[i] for i in gerador.permutation(len(blocos))]    # embaralha a ordem dos blocos

    n_teste = max(1, round(0.15 * len(blocos)))                       # ~15% dos blocos para teste
    n_val = max(1, round(0.15 * len(blocos)))                         # ~15% dos blocos para validação
    teste     += blocos[:n_teste]
    validacao += blocos[n_teste:n_teste + n_val]
    treino    += blocos[n_teste + n_val:]                             # o restante (~70%) para treino

idx_treino, idx_val, idx_teste = np.concatenate(treino), np.concatenate(validacao), np.concatenate(teste)
X_treino, y_treino = X[idx_treino], y[idx_treino]
X_val,    y_val    = X[idx_val],    y[idx_val]
X_teste,  y_teste  = X[idx_teste],  y[idx_teste]

print(f"{'integrante':>15} | treino | valid. | teste")
for c, nome in enumerate(CLASSES):
    print(f"{nome:>15} | {np.sum(y_treino == c):6d} | {np.sum(y_val == c):6d} | {np.sum(y_teste == c):5d}")

## 5. Aumento de dados

Com poucas fotos, a rede tende a **decorar** as imagens de treino (overfitting). Para evitar isso, criamos versões levemente modificadas de cada foto de treino, simulando situações reais:

| Transformação | Simula |
|---|---|
| Espelhamento horizontal | rosto virado para o outro lado |
| Deslocamento de até 8 pixels | rosto fora do centro |
| Contraste e brilho aleatórios | iluminação diferente |
| Ruído | câmera de qualidade pior |

O aumento é aplicado **apenas no treino**. Validação e teste usam as fotos reais.

In [ ]:
def aumentar(img):
    # Recebe uma imagem (96, 96, 1) com valores 0-1 e devolve uma versão modificada aleatoriamente

    # 1. Espelhamento horizontal (50% de chance)
    if np.random.rand() < 0.5:
        img = img[:, ::-1]

    # 2. Deslocamento: aumenta 8 px de borda (repetindo os pixels da beirada) e recorta numa posição sorteada
    dy, dx = np.random.randint(-8, 9, size=2)
    com_borda = np.pad(img, ((8, 8), (8, 8), (0, 0)), mode="edge")
    img = com_borda[8 + dy : 8 + dy + IMG_SIZE, 8 + dx : 8 + dx + IMG_SIZE]

    # 3. Contraste (multiplica a distância até o cinza médio) e brilho (soma um valor)
    img = (img - 0.5) * np.random.uniform(0.8, 1.2) + 0.5 + np.random.uniform(-0.1, 0.1)

    # 4. Ruído: soma um valor aleatório pequeno a cada pixel
    img = img + np.random.normal(0, 0.02, img.shape)

    return np.clip(img, 0, 1).astype(np.float32)    # garante que os valores fiquem entre 0 e 1

In [ ]:
# Imagens intermediárias do aumento de dados: a mesma foto modificada 6 vezes
fig, eixos = plt.subplots(N_CLASSES, 7, figsize=(13, 2 * N_CLASSES), squeeze=False)
for linha in range(N_CLASSES):
    original = X_treino[y_treino == linha][0]
    eixos[linha, 0].imshow(original[:, :, 0], cmap="gray", vmin=0, vmax=1)
    eixos[linha, 0].set_title(f"{CLASSES[linha]} (original)", fontsize=9)
    for coluna in range(1, 7):
        eixos[linha, coluna].imshow(aumentar(original)[:, :, 0], cmap="gray", vmin=0, vmax=1)
        eixos[linha, coluna].set_title(f"aumentada {coluna}", fontsize=9)
for ax in eixos.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Monta o treino final: as fotos originais + COPIAS_AUMENTADAS versões modificadas de cada uma
X_treino_final = [X_treino]
y_treino_final = [y_treino]
for _ in range(COPIAS_AUMENTADAS):
    X_treino_final.append(np.array([aumentar(img) for img in X_treino]))
    y_treino_final.append(y_treino)          # o rótulo não muda: continua sendo a mesma pessoa
X_treino_final = np.concatenate(X_treino_final)
y_treino_final = np.concatenate(y_treino_final)

print(f"Treino: {len(X_treino)} fotos originais -> {len(X_treino_final)} com o aumento de dados")

## 6. Montagem da CNN

```
Entrada 96x96x1
  Conv 3x3 (32 filtros)  + ReLU -> MaxPool 2x2  =>  48x48x32
  Conv 3x3 (64 filtros)  + ReLU -> MaxPool 2x2  =>  24x24x64
  Conv 3x3 (128 filtros) + ReLU -> MaxPool 2x2  =>  12x12x128
  Conv 3x3 (128 filtros) + ReLU -> MaxPool 2x2  =>   6x6x128
  Flatten                                        =>  4608 valores
  Dropout 50%
  Dense 128 + ReLU
  Dense (1 neurônio por integrante)              =>  pontuação de cada integrante
  Softmax                                        =>  probabilidade de cada integrante
```

- **Convolução:** filtros 3×3 deslizam pela imagem e respondem a padrões. As primeiras camadas detectam bordas; as últimas, partes do rosto. Os valores dos filtros são aprendidos no treino.
- **ReLU:** `max(0, x)`. Zera os valores negativos e dá à rede a capacidade de aprender relações não lineares.
- **MaxPooling 2×2:** fica com o maior valor de cada janela 2×2. Divide a resolução por 2 e tolera pequenos deslocamentos.
- **Flatten:** transforma os mapas 6×6×128 em um vetor de 4608 números.
- **Dropout:** desliga 50% dos neurônios aleatoriamente durante o treino, para a rede não depender de poucos neurônios.
- **Dense + Softmax:** a última camada densa dá uma pontuação para cada integrante. O softmax transforma essas pontuações em probabilidades que somam 1.

In [ ]:
# Cada linha recebe a saída da linha anterior (x) e passa adiante
entrada = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="entrada")                    # imagem 96x96, 1 canal

x = layers.Conv2D(32, 3, padding="same", activation="relu", name="conv1")(entrada)      # detecta bordas simples
x = layers.MaxPooling2D(2, name="pool1")(x)                                             # 96 -> 48

x = layers.Conv2D(64, 3, padding="same", activation="relu", name="conv2")(x)            # combina bordas em texturas
x = layers.MaxPooling2D(2, name="pool2")(x)                                             # 48 -> 24

x = layers.Conv2D(128, 3, padding="same", activation="relu", name="conv3")(x)           # partes do rosto
x = layers.MaxPooling2D(2, name="pool3")(x)                                             # 24 -> 12

x = layers.Conv2D(128, 3, padding="same", activation="relu", name="conv4")(x)           # combinações de partes
x = layers.MaxPooling2D(2, name="pool4")(x)                                             # 12 -> 6

x = layers.Flatten(name="flatten")(x)                                                   # 6x6x128 -> vetor de 4608
x = layers.Dropout(0.5, name="dropout")(x)                                              # combate o overfitting
x = layers.Dense(128, activation="relu", name="densa")(x)                               # combina as características
x = layers.Dense(N_CLASSES, name="pontuacoes")(x)                                       # 1 pontuação por integrante
saida = layers.Softmax(name="saida")(x)                                                 # pontuações -> probabilidades

model = keras.Model(entrada, saida)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),   # algoritmo que ajusta os pesos
    loss="sparse_categorical_crossentropy",                 # erro: entropia cruzada (rótulos como 0, 1, 2...)
    metrics=["accuracy"],                                   # acompanha a acurácia durante o treino
)

model.summary()   # mostra cada camada, o formato da saída e o número de parâmetros

## 7. Treinamento

Em cada **época**, a rede passa por todas as fotos de treino em lotes de 32. A cada lote:

1. Faz a previsão.
2. Calcula o erro com a **entropia cruzada**: `erro = −log(probabilidade dada à pessoa certa)`. Por exemplo, se a probabilidade certa foi 0,9, o erro é 0,1; se foi 0,1, o erro é 2,3.
3. Ajusta os pesos na direção que diminui o erro. Isso é feito por **backpropagation**, com o otimizador **Adam**.

O **EarlyStopping** observa o erro na validação. Se ele não melhorar por 10 épocas seguidas, o treino para e a rede volta aos pesos da melhor época.

In [ ]:
parada_antecipada = keras.callbacks.EarlyStopping(
    monitor="val_loss",            # observa o erro na validação
    patience=10,                   # tolera 10 épocas sem melhora
    restore_best_weights=True,     # no fim, volta para os pesos da melhor época
)

historico = model.fit(
    X_treino_final, y_treino_final,       # dados de treino (com aumento)
    validation_data=(X_val, y_val),       # dados de validação (fotos reais)
    epochs=EPOCAS,
    batch_size=BATCH,
    callbacks=[parada_antecipada],
    verbose=2,                            # imprime uma linha por época
)

In [ ]:
# Curvas de aprendizado: erro e acurácia a cada época
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))

eixos[0].plot(historico.history["loss"], label="treino")
eixos[0].plot(historico.history["val_loss"], label="validação")
eixos[0].set_title("Erro (entropia cruzada)")

eixos[1].plot(historico.history["accuracy"], label="treino")
eixos[1].plot(historico.history["val_accuracy"], label="validação")
eixos[1].set_title("Acurácia")

for ax in eixos:
    ax.set_xlabel("época")
    ax.legend()
    ax.grid(alpha=0.3)
plt.show()

**Como ler os gráficos:** o ideal é treino e validação caírem juntos. Se o erro de treino continua caindo enquanto o de validação sobe, a rede está decorando as fotos (**overfitting**).

## 8. Avaliação no conjunto de teste

### Matriz de confusão

As **linhas** são o integrante real e as **colunas** são o integrante previsto. A diagonal mostra os acertos e o resto mostra as confusões.

Para cada integrante *X*, a matriz fornece quatro contagens:
- **VP** (verdadeiro positivo): era X e a rede disse X. É a célula da diagonal.
- **FP** (falso positivo): a rede disse X, mas era outra pessoa. É o resto da coluna de X.
- **FN** (falso negativo): era X, mas a rede disse outra pessoa. É o resto da linha de X.
- **VN** (verdadeiro negativo): não era X e a rede não disse X. É todo o restante da matriz.

### Métricas

| Métrica | Fórmula | Pergunta que responde |
|---|---|---|
| Acurácia | acertos / total | Quantas fotos a rede acertou no total? |
| Precisão | VP / (VP + FP) | Quando a rede diz "é o X", quantas vezes acerta? |
| Recall | VP / (VP + FN) | Das fotos do X, quantas a rede reconheceu? |
| F1-Score | 2 · precisão · recall / (precisão + recall) | Equilíbrio entre precisão e recall |
| Especificidade | VN / (VN + FP) | Das fotos de outras pessoas, quantas não foram confundidas com o X? |

A **média macro** é a média simples entre os integrantes. Ela dá o mesmo peso a todos, mesmo a quem tem menos fotos.

In [ ]:
# Passo 1: a rede dá uma probabilidade para cada integrante; ficamos com a maior
probabilidades = model.predict(X_teste, verbose=0)
y_previsto = probabilidades.argmax(axis=1)

# Passo 2: monta a matriz de confusão contando cada par (real, previsto)
matriz = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
for real, previsto in zip(y_teste, y_previsto):
    matriz[real, previsto] += 1

# Passo 3: tira VP, FP, FN e VN da matriz (um valor por integrante)
VP = np.diag(matriz)                      # diagonal
FP = matriz.sum(axis=0) - VP              # soma da coluna menos a diagonal
FN = matriz.sum(axis=1) - VP              # soma da linha menos a diagonal
VN = matriz.sum() - VP - FP - FN          # todo o resto

# Passo 4: calcula as métricas (np.maximum evita divisão por zero)
precisao       = VP / np.maximum(VP + FP, 1)
recall         = VP / np.maximum(VP + FN, 1)
f1             = 2 * precisao * recall / np.maximum(precisao + recall, 1e-9)
especificidade = VN / np.maximum(VN + FP, 1)
acuracia       = VP.sum() / matriz.sum()

# Passo 5: mostra a tabela
print(f"{'integrante':>15} | precisão | recall |   F1  | especificidade | fotos")
for i, nome in enumerate(CLASSES):
    print(f"{nome:>15} |   {precisao[i]:.3f}  |  {recall[i]:.3f} | {f1[i]:.3f} |     {especificidade[i]:.3f}      | {matriz[i].sum()}")
print(f"{'média macro':>15} |   {precisao.mean():.3f}  |  {recall.mean():.3f} | {f1.mean():.3f} |     {especificidade.mean():.3f}")
print(f"\nAcurácia no teste: {acuracia:.3f}")

In [ ]:
# Desenho da matriz de confusão: quantidade e porcentagem da linha em cada célula
porcentagem = matriz / np.maximum(matriz.sum(axis=1, keepdims=True), 1)

tamanho = 2 + 1.2 * N_CLASSES
plt.figure(figsize=(tamanho, tamanho))
plt.imshow(porcentagem, cmap="Blues", vmin=0, vmax=1)
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        cor = "white" if porcentagem[i, j] > 0.5 else "black"
        plt.text(j, i, f"{matriz[i, j]}\n({porcentagem[i, j]:.0%})", ha="center", va="center", color=cor)
plt.xticks(range(N_CLASSES), CLASSES, rotation=30)
plt.yticks(range(N_CLASSES), CLASSES)
plt.xlabel("previsto")
plt.ylabel("real")
plt.title("Matriz de confusão (teste)")
plt.colorbar(fraction=0.046)
plt.show()

In [ ]:
# Métricas de cada integrante em um gráfico de barras
posicoes = np.arange(N_CLASSES)
plt.figure(figsize=(max(6, 2 * N_CLASSES), 4))
plt.bar(posicoes - 0.3, precisao, 0.2, label="Precisão")
plt.bar(posicoes - 0.1, recall, 0.2, label="Recall")
plt.bar(posicoes + 0.1, f1, 0.2, label="F1")
plt.bar(posicoes + 0.3, especificidade, 0.2, label="Especificidade")
plt.xticks(posicoes, CLASSES)
plt.ylim(0, 1.1)
plt.legend(ncol=4, loc="upper center")
plt.title("Métricas por integrante (teste)")
plt.show()

# Acurácia nos três conjuntos: se a do treino for bem maior que a do teste, há overfitting
for nome_conjunto, Xc, yc in [("treino", X_treino, y_treino), ("validação", X_val, y_val), ("teste", X_teste, y_teste)]:
    acc = np.mean(model.predict(Xc, verbose=0).argmax(axis=1) == yc)
    print(f"Acurácia em {nome_conjunto}: {acc:.3f}")

In [ ]:
# Fotos do teste que a rede errou
erros = np.where(y_previsto != y_teste)[0]
print(f"{len(erros)} erro(s) em {len(y_teste)} fotos de teste")

if len(erros) > 0:
    n = min(len(erros), 8)
    fig, eixos = plt.subplots(1, n, figsize=(2.3 * n, 2.8), squeeze=False)
    for ax, i in zip(eixos[0], erros[:n]):
        ax.imshow(X_teste[i][:, :, 0], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"real: {CLASSES[y_teste[i]]}\nprevisto: {CLASSES[y_previsto[i]]}", fontsize=9)
        ax.axis("off")
    plt.show()

## 9. Imagens intermediárias da rede

### Mapas de características

Mostramos a saída de cada camada convolucional para uma foto de teste. Cada quadradinho é a resposta de **um filtro**: as regiões claras são onde ele encontrou o padrão que procura.

- Em `conv1` (96×96), ainda dá para ver o rosto, porque os filtros detectam bordas.
- Nas camadas seguintes, a resolução cai e os mapas ficam mais abstratos, representando partes e combinações de partes do rosto.

In [ ]:
# Cria um "modelo espião" que devolve a saída das 4 camadas convolucionais
camadas = ["conv1", "conv2", "conv3", "conv4"]
espiao = keras.Model(entrada, [model.get_layer(nome).output for nome in camadas])

foto = X_teste[0]
saidas = espiao.predict(foto[np.newaxis], verbose=0)    # np.newaxis: a rede espera um lote de fotos

plt.figure(figsize=(2.5, 2.5))
plt.imshow(foto[:, :, 0], cmap="gray")
plt.title(f"entrada: {CLASSES[y_teste[0]]}")
plt.axis("off")
plt.show()

# Para cada camada, mostra os 8 primeiros mapas
fig, eixos = plt.subplots(len(camadas), 8, figsize=(14, 7.5))
for linha, (nome, mapas) in enumerate(zip(camadas, saidas)):
    for coluna in range(8):
        eixos[linha, coluna].imshow(mapas[0, :, :, coluna], cmap="viridis")
        eixos[linha, coluna].axis("off")
    eixos[linha, 0].set_title(f"{nome}: {mapas.shape[1]}x{mapas.shape[2]}, {mapas.shape[3]} mapas",
                              loc="left", fontsize=10)
plt.tight_layout()
plt.show()

### Onde a rede "olhou" (Grad-CAM)

O Grad-CAM gera um mapa de calor das regiões que mais pesaram na decisão:

1. Calculamos o quanto a pontuação da pessoa prevista muda quando cada mapa da `conv4` muda. Isso é o **gradiente**. Usamos a pontuação de antes do softmax porque, quando a rede tem quase 100% de certeza, a probabilidade quase não muda e o gradiente sai zerado.
2. A média do gradiente de cada mapa diz o quanto aquele mapa importa.
3. Somamos os mapas multiplicados pela sua importância e mantemos só os valores positivos.

Se o calor estiver no **rosto**, a rede aprendeu o que queríamos. Se estiver no **fundo**, ela está usando o cenário para decidir, e precisamos de fotos em mais lugares diferentes.

In [ ]:
# Modelo auxiliar que devolve, ao mesmo tempo, os mapas da conv4 e as pontuações (antes do softmax)
modelo_gradcam = keras.Model(entrada, [model.get_layer("conv4").output, model.get_layer("pontuacoes").output])

def grad_cam(img):
    with tf.GradientTape() as fita:                              # "grava" as contas para calcular gradientes
        mapas, pontuacoes = modelo_gradcam(img[np.newaxis])
        classe = int(tf.argmax(pontuacoes[0]))                   # integrante previsto
        nota = pontuacoes[0, classe]                             # pontuação dele
    gradientes = fita.gradient(nota, mapas)                      # quanto a nota muda com cada valor dos mapas
    importancia = tf.reduce_mean(gradientes, axis=(0, 1, 2))     # importância média de cada mapa
    calor = tf.nn.relu(tf.reduce_sum(mapas[0] * importancia, axis=-1)).numpy()   # soma ponderada (só positivos)
    calor = calor / (calor.max() + 1e-9)                         # escala para 0-1
    return redimensionar(calor, IMG_SIZE), classe                # amplia de 12x12 para 96x96

# Uma foto de teste de cada integrante: imagem original em cima, mapa de calor embaixo
fig, eixos = plt.subplots(2, N_CLASSES, figsize=(3 * N_CLASSES, 6), squeeze=False)
for c in range(N_CLASSES):
    img = X_teste[y_teste == c][0]
    calor, previsto = grad_cam(img)
    eixos[0, c].imshow(img[:, :, 0], cmap="gray")
    eixos[0, c].set_title(f"real: {CLASSES[c]}")
    eixos[1, c].imshow(img[:, :, 0], cmap="gray")
    eixos[1, c].imshow(calor, cmap="jet", alpha=0.45, interpolation="bilinear")   # mapa de calor por cima da foto
    eixos[1, c].set_title(f"previsto: {CLASSES[previsto]}")
for ax in eixos.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

## 10. Salvar o modelo e testar com fotos novas

A foto nova passa **exatamente** pelo mesmo `preprocessar` usado no treino. Se a maior probabilidade ficar abaixo de 60%, dizemos que a pessoa é "desconhecida".

In [ ]:
# Salva a rede treinada no Drive (pode ser carregada depois com keras.models.load_model)
caminho_modelo = os.path.join(os.path.dirname(DATASET_DIR), "modelo_cnn.keras")
model.save(caminho_modelo)
print("Modelo salvo em:", caminho_modelo)

In [ ]:
def reconhecer(img):
    x = preprocessar(img)                                      # mesmo tratamento do treino
    probs = model.predict(x[np.newaxis], verbose=0)[0]         # probabilidade de cada integrante
    melhor = probs.argmax()
    resposta = CLASSES[melhor] if probs[melhor] >= 0.6 else "desconhecido"

    fig, eixos = plt.subplots(1, 3, figsize=(12, 3), gridspec_kw={"width_ratios": [1, 1, 1.5]})
    eixos[0].imshow(img)
    eixos[0].set_title("foto")
    eixos[1].imshow(x[:, :, 0], cmap="gray")
    eixos[1].set_title("entrada da CNN")
    eixos[2].barh(CLASSES, probs)
    eixos[2].set_xlim(0, 1)
    eixos[2].set_title(f"Resposta: {resposta} ({probs[melhor]:.0%})")
    eixos[0].axis("off")
    eixos[1].axis("off")
    plt.tight_layout()
    plt.show()

# Exemplo com uma foto do conjunto de teste
reconhecer(ler_imagem(caminhos[idx_teste[0]]))

In [ ]:
# Foto nova por upload
TESTAR_UPLOAD = False   # troque para True e rode a célula

if EM_COLAB and TESTAR_UPLOAD:
    from google.colab import files
    for nome_arquivo, conteudo in files.upload().items():
        reconhecer(np.array(Image.open(io.BytesIO(conteudo)).convert("RGB")))

### Reconhecimento ao vivo pela webcam

A webcam fica ligada e a rede tenta reconhecer quem está na frente dela **continuamente**, até clicarmos em **Parar webcam**. A cada repetição:

1. O navegador desenha sobre o vídeo as caixas e os nomes do resultado anterior.
2. Envia ao Python o quadro (frame) atual do vídeo.
3. O Python recorta cada região, aplica o **mesmo** `preprocessar` do treino e passa pela CNN.
4. O Python devolve os nomes, e o ciclo recomeça (de 3 a 5 vezes por segundo).

**Mais de uma pessoa:** o trabalho não permite usar um detector de rosto pronto, então a rede não sabe *onde* há rostos na imagem. A solução simples é dividir a tela em regiões fixas, lado a lado (`N_PESSOAS = 2` gera duas caixas), e cada pessoa se posiciona dentro de uma caixa.

Com `N_PESSOAS = 1`, a caixa é exatamente o recorte usado nas fotos de treino, e é quando a rede funciona melhor. Com 2 caixas, cada uma é menor, então as pessoas devem ficar um pouco mais perto da câmera.

Para evitar que o nome fique "piscando", usamos a **média das probabilidades dos últimos 5 quadros**.

In [ ]:
# Código JavaScript: mostra a webcam ao vivo, desenha os resultados e entrega quadros ao Python
JS_AO_VIVO = '''
var video = null, tela = null, area = null, parar = false;

async function iniciarWebcam() {
  // 1. Monta na página: botão "Parar", vídeo da câmera e uma tela transparente por cima (para as caixas)
  area = document.createElement('div');
  const botao = document.createElement('button');
  botao.textContent = 'Parar webcam';
  botao.onclick = () => { parar = true; };
  const moldura = document.createElement('div');
  moldura.style.position = 'relative';
  video = document.createElement('video');
  video.style.display = 'block';
  tela = document.createElement('canvas');
  tela.style.position = 'absolute';
  tela.style.left = '0';
  tela.style.top = '0';
  moldura.append(video, tela);
  area.append(botao, moldura);
  document.body.appendChild(area);

  // 2. Liga a webcam; a tela de desenho fica do mesmo tamanho do vídeo
  const stream = await navigator.mediaDevices.getUserMedia({video: true});
  video.srcObject = stream;
  await video.play();
  tela.width = video.videoWidth;
  tela.height = video.videoHeight;
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
}

async function proximoQuadro(caixas) {
  // Chamada pelo Python a cada repetição. "caixas" = lista de [x, y, lado, texto, cor]
  if (video === null) await iniciarWebcam();

  // 3. Se clicaram em "Parar": desliga a câmera e devolve null (o Python entende que acabou)
  if (parar) {
    video.srcObject.getVideoTracks()[0].stop();
    area.remove();
    return null;
  }

  // 4. Desenha cada caixa com o nome (resultado calculado pelo Python)
  const ctx = tela.getContext('2d');
  ctx.clearRect(0, 0, tela.width, tela.height);
  ctx.lineWidth = 3;
  ctx.font = 'bold 22px sans-serif';
  for (const [x, y, lado, texto, cor] of caixas) {
    ctx.strokeStyle = cor;
    ctx.strokeRect(x, y, lado, lado);                                  // contorno da caixa
    ctx.fillStyle = cor;
    ctx.fillRect(x, y, ctx.measureText(texto).width + 12, 30);         // fundo do texto
    ctx.fillStyle = 'white';
    ctx.fillText(texto, x + 6, y + 23);                                // nome + confiança
  }

  // 5. Captura o quadro atual do vídeo e devolve ao Python como JPEG em texto (base64)
  const captura = document.createElement('canvas');
  captura.width = video.videoWidth;
  captura.height = video.videoHeight;
  captura.getContext('2d').drawImage(video, 0, 0);
  return captura.toDataURL('image/jpeg', 0.8);
}
'''

In [ ]:
def regioes_da_tela(largura, altura, n_pessoas):
    # Divide a tela em n faixas verticais e pega o maior quadrado no centro de cada faixa.
    # Com 1 pessoa, o quadrado é o mesmo recorte central usado nas fotos de treino.
    faixa = largura // n_pessoas
    lado = min(faixa, altura)
    caixas = []
    for i in range(n_pessoas):
        x = i * faixa + (faixa - lado) // 2       # centraliza o quadrado dentro da faixa
        y = (altura - lado) // 2
        caixas.append((x, y, lado))
    return caixas


def reconhecimento_ao_vivo(n_pessoas=1, limiar=0.6, quadros_na_media=5):
    from google.colab.output import eval_js
    from IPython.display import display, Javascript
    import json

    display(Javascript(JS_AO_VIVO))                      # envia o código JavaScript para o navegador
    ultimas_probs = [[] for _ in range(n_pessoas)]       # probabilidades dos últimos quadros de cada caixa
    desenho = []                                         # caixas e nomes a desenhar (começa vazio)

    while True:
        # 1. Manda o desenho do resultado anterior e recebe um quadro novo da webcam
        resposta = eval_js(f"proximoQuadro({json.dumps(desenho)})")
        if resposta is None:                             # clicaram em "Parar webcam"
            break
        quadro = np.array(Image.open(io.BytesIO(base64.b64decode(resposta.split(",")[1]))).convert("RGB"))
        altura, largura = quadro.shape[:2]
        caixas = regioes_da_tela(largura, altura, n_pessoas)

        # 2. Recorta cada caixa e aplica o MESMO pré-processamento do treino
        entradas = np.array([preprocessar(quadro[y:y + lado, x:x + lado]) for x, y, lado in caixas])

        # 3. Passa todas as caixas pela CNN de uma vez
        probs = model(entradas, training=False).numpy()

        # 4. Para cada caixa: média dos últimos quadros, decide o nome e prepara o desenho
        desenho = []
        for i, (x, y, lado) in enumerate(caixas):
            ultimas_probs[i] = (ultimas_probs[i] + [probs[i]])[-quadros_na_media:]   # guarda só os últimos N
            media = np.mean(ultimas_probs[i], axis=0)
            melhor = int(media.argmax())
            if media[melhor] >= limiar:
                texto, cor = f"{CLASSES[melhor]} ({media[melhor]:.0%})", "#2ca02c"   # verde: reconhecido
            else:
                texto, cor = "desconhecido", "#d62728"                               # vermelho: incerto
            desenho.append([int(x), int(y), int(lado), texto, cor])

    print("Webcam desligada.")

In [ ]:
# Reconhecimento ao vivo
TESTAR_AO_VIVO = False   # troque para True e rode a célula
N_PESSOAS = 1            # 1 = uma caixa central | 2 = duas caixas lado a lado

if EM_COLAB and TESTAR_AO_VIVO:
    reconhecimento_ao_vivo(n_pessoas=N_PESSOAS)

## 11. Perguntas prováveis na apresentação

**Dados e pré-processamento**
- *Por que recortar em vez de só redimensionar?* Redimensionar uma foto retangular para quadrado achataria o rosto.
- *Por que tons de cinza?* A identidade está na forma do rosto, não na cor. Com 1 canal a rede fica menor e depende menos da iluminação.
- *Por que a média de blocos antes de reduzir?* Pegar só 1 pixel a cada 30 gera serrilhado. A média resume todos os pixels do bloco.
- *O que a equalização faz?* Usa o histograma acumulado para espalhar os tons de cinza de 0 a 255, deixando fotos claras e escuras parecidas.
- *Por que dividir em blocos de fotos seguidas?* Fotos seguidas são quase iguais. Separá-las ao acaso colocaria "gêmeas" no treino e no teste, inflando a nota.
- *Por que aumento de dados só no treino?* O teste precisa medir o desempenho em fotos reais.

**A rede**
- *Por que CNN e não uma rede comum (densa)?* O mesmo filtro percorre a imagem inteira, então a rede precisa de muito menos parâmetros e reconhece o padrão em qualquer posição.
- *Quantos parâmetros tem uma convolução?* `3 · 3 · canais_de_entrada · filtros + filtros` (o último termo é o bias). Por exemplo, `conv2` tem 3·3·32·64 + 64 = 18 496. Os números aparecem no `model.summary()`.
- *O que faz o pooling?* Reduz a imagem pela metade (menos contas) e torna a rede tolerante a pequenos deslocamentos.
- *Para que serve o dropout?* Desligar neurônios ao acaso obriga a rede a não depender de poucos deles, o que reduz o overfitting.
- *Por que softmax?* Ele transforma as pontuações da última camada em probabilidades que somam 1, uma para cada integrante.
- *O que são época e batch?* Época é uma passada completa pelo treino. Batch é o grupo de fotos usado em cada ajuste dos pesos.
- *Como a rede aprende?* Ela calcula o erro, o backpropagation calcula quanto cada peso contribuiu para ele, e o Adam ajusta os pesos para reduzir o erro.

**Métricas**
- *Precisão × recall?* A precisão olha a **coluna** da matriz: das vezes que a rede disse X, quantas eram X. O recall olha a **linha**: das fotos de X, quantas a rede acertou.
- *Por que não só a acurácia?* Se uma pessoa tiver muito mais fotos, a acurácia pode ser alta mesmo errando as outras.
- *Limitações:* sem detector de rosto, o rosto precisa estar no centro da foto. Além disso, o softmax sempre escolhe alguém, por isso usamos o limite de 60% para "desconhecido".
- *Como reconhece várias pessoas ao vivo?* Sem detector de rosto, a rede não sabe onde estão os rostos. Por isso a tela é dividida em caixas fixas e cada pessoa se posiciona em uma. Achar rostos em qualquer lugar exigiria um detector (como os do OpenCV), que o trabalho não permite.